# Логистическая регрессия. Практика

В этом задании вам предлагается спрогнозировать, купит клиент велосипед или нет, обучив логистическую регрессию.

In [21]:
# подключим библиотеки
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [22]:
# считаем данные
data = pd.read_csv('https://raw.githubusercontent.com/evgpat/edu_stepik_practical_ml/main/datasets/bike_buyers_clean.csv')

In [23]:
# выводим первые 5 строк датафрейма
data.head(5)

,ID,Marital Status,Gender,Income,Children,Education,Occupation,Home Owner,Cars,Commute Distance,Region,Age,Purchased Bike
0,12496,Married,Female,40000,1,Bachelors,Skilled Manual,Yes,0,0-1 Miles,Europe,42,No
1,24107,Married,Male,30000,3,Partial College,Clerical,Yes,1,0-1 Miles,Europe,43,No
2,14177,Married,Male,80000,5,Partial College,Professional,No,2,2-5 Miles,Europe,60,No
3,24381,Single,Male,70000,0,Bachelors,Professional,Yes,1,5-10 Miles,Pacific,41,Yes
4,25597,Single,Male,30000,0,Bachelors,Clerical,No,0,0-1 Miles,Europe,36,Yes


In [24]:
# смотрим размер датафрейма
data.shape

(1000, 13)

Выведите статистики по категориальным признакам, чтобы посмотреть, сколько категорий в каждом категориальном (нечисловом) признаке.

Для этого можно воспользоваться методом `describe` из библиотеки pandas со значением параметра `include = 'object'`.

**Вопрос:** в каком категориальном признаке встречаются три различных значения?

In [25]:
data.describe(include='object')

,Marital Status,Gender,Education,Occupation,Home Owner,Commute Distance,Region,Purchased Bike
count,1000,1000,1000,1000,1000,1000,1000,1000
unique,2,2,5,5,2,5,3,2
top,Married,Male,Bachelors,Professional,Yes,0-1 Miles,North America,No
freq,539,509,306,276,685,366,508,519


Закодируйте все категориальные столбцы с двумя категориями следующим образом:  
самая часто встречающаяся категория превращается в 1, другая в 0.

In [26]:
cat_cols = data.select_dtypes(['object'])
binary_cat_cols = cat_cols.loc[:, cat_cols.nunique() == 2]
modes = binary_cat_cols.mode().iloc[0]
binary_cat_cols = (binary_cat_cols == modes).astype(int)
binary_cat_cols

,Marital Status,Gender,Home Owner,Purchased Bike
0,1,0,1,1
1,1,1,1,1
2,1,1,0,1
3,0,1,1,0
4,0,1,0,0
...,...,...,...,...
995,1,1,1,0
996,0,1,1,0
997,1,1,1,0
998,0,1,0,1


In [27]:
data[binary_cat_cols.columns] = binary_cat_cols

Удалите остальные категориальные столбцы.

**Вопрос:** сколько категориальных столбцов вы удалили?

In [33]:
non_binary_cat_cols = cat_cols.loc[:, cat_cols.nunique() != 2]
data = data.drop(non_binary_cat_cols.columns, axis=1)

Удалите столбец `ID`, так как он по сути нечисловой.

In [34]:
data = data.drop('ID', axis=1)

In [35]:
data

,Marital Status,Gender,Income,Children,Home Owner,Cars,Age,Purchased Bike
0,1,0,40000,1,1,0,42,1
1,1,1,30000,3,1,1,43,1
2,1,1,80000,5,0,2,60,1
3,0,1,70000,0,1,1,41,0
4,0,1,30000,0,0,0,36,0
...,...,...,...,...,...,...,...,...
995,1,1,60000,2,1,2,54,0
996,0,1,70000,4,1,0,35,0
997,1,1,60000,2,1,0,38,0
998,0,1,100000,3,0,3,38,1


Сформируйте матрицу объект-признак `X` и вектор `y` с целевой переменной.  
Целевая переменная - это последний столбец, `Purchased Bike`.

In [36]:
X = data.drop('Purchased Bike', axis=1)
y = data['Purchased Bike']

Разбейте данные на тренировочную и тестовую часть (`Xtrain`, `Xtest`, `ytrain`, `ytest`), в тест отправьте 30% данных.  
Зафиксируйте `random_state = 42`.

In [37]:
from sklearn.model_selection import train_test_split

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=42)

**Вопрос:** сколько объектов в матрице `Xtrain`?

In [38]:
Xtrain.shape

(700, 7)

Почти всё готово для обучения модели!

Осталось отмасштабировать матрицу `X`, так как линейные модели чувствительны к масштабу данных.

*  Обучите на тренировочной матрице (`Xtrain`) `MinMaxScaler` из библиотеки `sklearn.preprocessing`
*  Примените масштабирование и к `Xtrain`, и к `Xtest`
*  Переведите полученные после масштабирования `np.array` обратно в pandas `dataframe`.

Полученные масштабированные матрицы назовите, как и раньше, `Xtrain` и `Xtest`.

In [39]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

Xtrain = pd.DataFrame(scaler.fit_transform(Xtrain), columns=Xtrain.columns)
Xtest = pd.DataFrame(scaler.transform(Xtest), columns=Xtest.columns)

Теперь обучите логистическую регрессию на тренировочных данных

In [40]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(Xtrain, ytrain)

LogisticRegression()

Сделайте предсказания на тренировочных и на тестовых данных.

In [41]:
pred_train = lr.predict(Xtrain)
pred_test = lr.predict(Xtest)

Оцените значение accuracy на трейне и на тесте.

In [42]:
from sklearn.metrics import accuracy_score

print('train: ', accuracy_score(ytrain, pred_train))
print('test: ', accuracy_score(ytest, pred_test))

train:  0.6342857142857142
test:  0.5766666666666667


Качество модели получилось невысоким, зато модель не переобучена.

Попробуем добавить новых признаков в модель, используя [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

Создайте полиномиальные признаки degree = 2.

Как обычно:
*  `fit` делайте на тренировочных данных
*  `transform` и на тренировочных, и на тестовых данных. Затем верните результат к формату pandas `dataframe`.

Полученные матрицы назовите, как и раньше, `Xtrain` и `Xtest`.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=2)

Xtrain = pd.DataFrame(pf.fit_transform(Xtrain))
Xtest = pd.DataFrame(pf.transform(Xtest))

In [45]:
Xtrain

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,1.0,1.0,1.0,0.2500,0.2,1.0,0.00,0.140625,1.0,1.0,...,0.04,0.2,0.00,0.028125,1.0,0.00,0.140625,0.0000,0.000000,0.019775
1,1.0,0.0,1.0,0.5000,0.0,0.0,0.75,0.140625,0.0,0.0,...,0.00,0.0,0.00,0.000000,0.0,0.00,0.000000,0.5625,0.105469,0.019775
2,1.0,0.0,1.0,0.1875,0.0,0.0,0.00,0.171875,0.0,0.0,...,0.00,0.0,0.00,0.000000,0.0,0.00,0.000000,0.0000,0.000000,0.029541
3,1.0,0.0,1.0,0.6250,0.0,0.0,0.75,0.109375,0.0,0.0,...,0.00,0.0,0.00,0.000000,0.0,0.00,0.000000,0.5625,0.082031,0.011963
4,1.0,1.0,1.0,0.5000,1.0,1.0,0.75,0.250000,1.0,1.0,...,1.00,1.0,0.75,0.250000,1.0,0.75,0.250000,0.5625,0.187500,0.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,1.0,1.0,1.0,0.3750,0.4,1.0,0.50,0.421875,1.0,1.0,...,0.16,0.4,0.20,0.168750,1.0,0.50,0.421875,0.2500,0.210938,0.177979
696,1.0,0.0,0.0,0.0000,0.4,1.0,0.00,0.406250,0.0,0.0,...,0.16,0.4,0.00,0.162500,1.0,0.00,0.406250,0.0000,0.000000,0.165039
697,1.0,0.0,1.0,0.1250,0.0,1.0,0.25,0.109375,0.0,0.0,...,0.00,0.0,0.00,0.000000,1.0,0.25,0.109375,0.0625,0.027344,0.011963
698,1.0,0.0,0.0,0.0000,0.4,0.0,0.25,0.671875,0.0,0.0,...,0.16,0.0,0.10,0.268750,0.0,0.00,0.000000,0.0625,0.167969,0.451416


**Вопрос:** на сколько признаков стало больше при добавлении полиномиальных признаков второй степени?

Заново обучите логистическую регрессию, уже на расширенной матрице признаков, и сделайте предсказания на трейне и тесте, а затем оцените качество (*accuracy*).

In [47]:
lr2 = LogisticRegression()

lr2.fit(Xtrain, ytrain)
pred2_train = lr2.predict(Xtrain)
pred2_test = lr2.predict(Xtest)

print('train: ', accuracy_score(ytrain, pred2_train))
print('test: ', accuracy_score(ytest, pred2_test))

train:  0.6857142857142857
test:  0.6233333333333333


**Вопрос:** на сколько повысилось качество модели на тестовых данных?  
Ответ округлите до сотых.

Появились новые требования от заказчика!

Заказчик просит, чтобы:
*  доля найденных моделью потенциальных покупателей была максимальной
*  accuracy при этом была не ниже, чем 0.6 (отклонения *accuracy* на тестовых данных на $\pm 0.05$ допустимы).

Сначала посмотрите, какие значения *recall* и *accuracy* имеют предсказния модели на тесте с классами, предсказанными по умолчанию (методом `predict`).

In [48]:
from sklearn.metrics import recall_score

print('recall: ', recall_score(ytest, pred2_test))
print('accuracy: ', accuracy_score(ytest, pred2_test))

recall:  0.6959459459459459
accuracy:  0.6233333333333333


Подберём порог для перевода вероятностей в классы, чтобы оптимизировать требуемые метрики!

Разобъем тренировочные данные на трейн и валидацию, чтобы по валидационной части подбирать порог.

In [49]:
XtrainS, Xval, ytrainS, yval = train_test_split(Xtrain, ytrain, test_size=0.3, random_state=42)

XtrainS.shape, Xval.shape

((490, 36), (210, 36))

* Обучите модель на тренировочных данных.
* Предскажите вероятности положительного класса на валидационных данных

В цикле для каждого значения порога:
*  переведите вероятности в классы
*  вычислите полноту (на валидационных данных)

Выведите на экран:

1) значение порога, дающее максимальный *recall*, при условии, что *accuracy* $\geq$ 0.6.

2) значение *recall* при этом пороге

3) значение *accuracy* для этого порога


Ищите порог на отрезке от 0 до 1 с шагом 0.01.

In [ ]:
lr3 = LogisticRegression()

lr3.fit(XtrainS, ytrainS)
pred_proba_val = lr3.predict_proba(Xval)


RecMax = -1
BestThr = -1
BestAcc = -1

for thr in np.arange(0, 1, 0.01):
    curr_res = (pred_proba_val[:, 1] >= thr).astype(int)
    rec = recall_score(yval, curr_res)
    acc = accuracy_score(yval, curr_res)
    if rec > RecMax and acc >= 0.6:
        RecMax = rec
        BestAcc = acc
        BestThr = thr

print(BestThr, RecMax, BestAcc)

0.42 0.8518518518518519 0.6142857142857143


Теперь заново обучите модель на исходных тренировочных данных (`Xtrain`, `ytrain`), предскажите вероятности на тесте и переведите их в классы по найденному порогу.

In [51]:
lr4 = LogisticRegression()

lr4.fit(Xtrain, ytrain)
final_pred_proba = lr4.predict_proba(Xtest)

final_res = (final_pred_proba[:, 1] >= BestThr).astype(int)

print('recall: ', recall_score(ytest, final_res))
print('accuracy: ', accuracy_score(ytest, final_res))

recall:  0.8040540540540541
accuracy:  0.5666666666666667


**Вопрос:** какое значение *recall* получилось на тестовых данных после подбора порога?  
Ответ округлите до десятых.

При помощи подбора порога удалось сильно увеличить значение *recall*!  
Однако, как видно, на тесте не удалось сохранить условие $accuracy \geq 0.6$ (но в допустимые рамки уложились!)

Это свидетельство небольшого переобучения модели. Однако в сухом остатке имеет значительное увеличение полноты, что является приоритетом для заказчика.